In [1]:
# https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html
%load_ext autoreload
%autoreload 2

# 面向对象编程：核心知识点速览

本章将带你快速掌握面向对象编程（Object-Oriented Programming, 简称OOP）的核心概念与 Python 实现方式。主要内容如下：

1. 面向过程 vs. 面向对象：先通过面向过程的方式实现银行账户管理，理解其局限，引出面向对象的必要性与优势。  
2. 类与对象：从“类是蓝图、对象是实例”出发，学习字段与方法的定义、实例化过程以及 `self` 的含义。  
3. 封装与继承：  
   - 封装：通过访问控制（公有/私有）与属性装饰器保护数据。  
   - 继承：掌握单继承、`super()` 的正确用法，理解代码复用的基本逻辑。    
4. 进阶语法：掌握一些常见的特殊方法，如 `__new__`、`__repr__` 等，让类的行为更符合预期。

阅读完本章，你将能够独立写出可扩展、可维护的 Python 面向对象代码，为后续设计模式与大型项目架构打下坚实基础。


## 第一部分：面向过程编程

### 1.1 面向过程的银行账户管理

下面的代码将用面向过程方式实现一个简易银行账户管理系统，支持开户、存取款及查询余额。通过函数与全局字典配合，演示无封装状态下数据暴露与功能分散带来的维护难题。

In [2]:
# ========== 面向过程版本：银行账户管理系统 ==========
import datetime

# 辅助函数：获取当前时间
def get_current_time():    
    return datetime.datetime.now()

# 1. 创建账户
def create_account(name, initial_deposit, accounts, timestamp=None):
    """创建一个新账户"""
    
    if initial_deposit < 0:
        print("初始存款不能为负数")
        return None

    # 生成新账户编号，格式为001, 002, ..., 999
    account_num = f"{len(accounts) + 1:03d}"

    account = {
        'name': name,
        'balance': initial_deposit,
        'transactions': [
            {
                'time': timestamp if timestamp else get_current_time(),
                'type': '开户',
                'amount': initial_deposit,
                'balance': initial_deposit
            }
        ]
    }
    
    # 将新账户添加到账户字典中
    accounts[account_num] = account

    print(f"账户 {account_num} 创建成功，户主：{name}")
    
    return account

# 2. 存款函数
def deposit(account, amount, timestamp=None):
    """存款"""
    
    if amount <= 0:
        print("错误：存款金额必须大于0")
        return False
    
    account['balance'] += amount
    account['transactions'].append({
        'time': timestamp if timestamp else get_current_time(),
        'type': '存款',
        'amount': amount,
        'balance': account['balance']
    })
    
    print(f"账户 {account['name']} 存款成功，金额：{amount}，余额：{account['balance']}")
    return True

# 3. 取款函数
def withdraw(account, amount, timestamp=None):
    """取款"""
    if amount <= 0:
        print("取款金额必须大于0")
        return False
    
    if account['balance'] < amount:
        print("余额不足")
        return False
    
    account['balance'] -= amount
    account['transactions'].append({
        'time': timestamp if timestamp else get_current_time(),
        'type': '取款',
        'amount': -amount,
        'balance': account['balance']
    })
    
    print(f"账户 {account['name']} 取款成功，金额：{amount}，余额：{account['balance']}")
    return True

# 4. 查询余额
def get_balance(account):
    """查询余额"""
    return account['balance']
    
# 5. 查询交易记录
def display_transactions(account, limit=None):
    """查询最近交易记录"""
    transactions = account['transactions']
    if limit:
        transactions = transactions[-limit:]
    for trans in transactions:
        print(f"  {trans['time'].strftime("%Y-%m-%d %H:%M:%S")} - {trans['type']}: {trans['amount']}, 余额: {trans['balance']}")

In [3]:
# 全局变量：存储所有账户
accounts = {}

# 使用示例
print("===== 面向过程版本 =====")
acc1 = create_account("张三", 1000.0, accounts)
acc2 = create_account("李四", 500.0, accounts)

if acc1:
    deposit(acc1, 500.50)
    withdraw(acc1, 200.25)
    print(f"张三余额: {get_balance(acc1)}")
    
    # 显示交易记录
    print("最近交易记录:")
    display_transactions(acc1, 3)

===== 面向过程版本 =====
账户 001 创建成功，户主：张三
账户 002 创建成功，户主：李四
账户 张三 存款成功，金额：500.5，余额：1500.5
账户 张三 取款成功，金额：200.25，余额：1300.25
张三余额: 1300.25
最近交易记录:
  2026-01-08 00:35:04 - 开户: 1000.0, 余额: 1000.0
  2026-01-08 00:35:04 - 存款: 500.5, 余额: 1500.5
  2026-01-08 00:35:04 - 取款: -200.25, 余额: 1300.25



### 1.2 问题分析

#### 问题1：数据与操作分离

没有把”这是一个账户“的概念明确表达出来，导致对账户数据的操作分散在不同的函数中。
+ 与”账户“这一概念有关的”数据“在`create_account`函数中被打包为具有特殊结构的字典对象`account`
+ 但是对这些”数据“的”操作“却被分散的被定义在不同的函数中（`deposit`, `withdraw`, `get_balance`, `display_transactions`）
 

#### 问题2：数据可以被任意修改

因为账户数据只是被打包为一个普通的字典对象，所以可以被任意修改。

```python
# 危险操作1：可以直接修改余额
acc1['balance'] = 1000000  # 没有验证，没有记录交易

# 危险操作2：可以直接修改交易记录
acc1['transactions'].append({
    'time': '2024-01-01 00:00:00',
    'type': '虚假存款',
    'amount': 50000,
    'balance': 60000
})

# 危险操作3：可以绕过所有验证逻辑
def malicious_deposit(account, amount):
    """恶意函数，绕过所有验证"""
    account['balance'] += amount
    # 不记录交易，不留痕迹
```


## 第二部分：面向对象编程：基础

### 2.1 什么是类？为什么要用类？

**类的核心思想**：将**数据**和**操作数据的函数**捆绑在一起，形成一个完整的实体。

**现实世界类比**：
- 银行账户 = 数据（账户号、姓名、余额、交易记录） + 操作（存款、取款、查询）
- 汽车 = 数据（品牌、颜色、速度） + 操作（启动、加速、刹车）
- 学生 = 数据（学号、姓名、成绩） + 操作（选课、考试、计算GPA）

**类的定义**：类是**自定义的数据类型**。它定义了这类数据应该有哪些组成部分（字段）和能进行哪些操作（方法）。

**类与对象的关系**：
- 类（Class）是“模板”或“蓝图”，它定义了数据结构和行为。
- 对象（Object）是“实物”，是根据类的模板构造出的具体数据。

以银行账户为例，类定义了所有账户的共有数据（账户号、余额）和操作（存款、取款），而对象则是具体的账户实例，每个对象都有自己的账户号和余额，可以进行独立于其它账户的存款和取款操作。

**对象（Object）与实例（Instance）的关系**：
- “实例”是“对象”的同义词，强调“由类实例化而来”。
- 常见表达：“张三的银行账户”是“银行账户类”的一个对象/实例/对象实例，通过对“银行账户类”实例化（instantiate）得到。

### 2.2 定义第一个类：银行账户

In [2]:
# ========== 面向对象版本：银行账户类 ==========

import datetime

class Account:
    """
    银行账户类
    定义了"银行账户"这种数据类型
    """
    
    # ========== 第一部分：字段（Field/属性）==========
    # 字段是类中存储数据的变量
    
    # 1. 类字段（所有账户共享）
    bank_name = "中国银行"  # 所有账户都属于同一银行
    
    # 2. 实例字段（每个账户独立）
    # 这些在 __init__ 方法中定义
    
    # ========== 第二部分：方法（Method）==========
    # 方法是类中定义的函数，用于操作数据
    
    def __init__(self, name, accounts, initial_deposit=0.0, timestamp=None):
        """
        初始化方法（构造函数）
        当创建 Account 对象时自动调用
        
        参数说明：
        - self: 表示对象自身（重要！）
        - name: 户主姓名
        - accounts: 账户列表
        - initial_deposit: 初始存款金额
        """
        
        # ========== 这里定义实例字段 ==========
        
        # 1. 基本账户信息
        self.account_num = f"{len(accounts) + 1:03d}"  # 账户号
        self.name = name  # 户主姓名
        
        # 2. 余额相关（使用私有属性保护）
        self.__balance = 0.0  # 私有属性，外部不能直接访问
        
        # 3. 交易记录（列表存储）
        self.__transactions = []  # 私有属性，外部不能直接访问
        
        # ========== 初始化逻辑 ==========
        
        # 验证并设置初始存款
        if initial_deposit < 0:
            raise ValueError("初始存款不能为负数")
        
        self.__balance = initial_deposit # 私有属性，外部不能直接访问
        
        # 记录开户交易
        self.__record_transaction("开户", initial_deposit, timestamp)
        
        # 将新账户添加到账户字典中
        accounts[self.account_num] = self

        print(f"账户创建成功: {self.name} ({self.account_num})")
    
    # ========== 私有方法（外部无法直接调用）==========
    def __record_transaction(self, trans_type, amount, timestamp):
        """
        私有方法：记录交易
        外部不能直接调用，只能在类内部使用
        """
        transaction = {
            'time': timestamp if timestamp else datetime.datetime.now(),
            'type': trans_type,
            'amount': amount,
            'balance': self.__balance
        }
        self.__transactions.append(transaction)
    
    # ========== 公共方法（外部可以调用）==========
    def get_balance(self):
        """查询余额"""
        return self.__balance

    def deposit(self, amount, timestamp=None):
        """
        存款方法
        
        参数：
        - amount: 存款金额
        
        返回：
        - 成功返回 True，失败返回 False
        """
        
        # 验证1：金额必须为正数
        if amount <= 0:
            print("错误：存款金额必须大于0")
            return False
        
        # 执行存款
        self.__balance += amount
        
        # 记录交易
        self.__record_transaction("存款", amount, timestamp)
        
        print(f"存款成功！余额: {self.__balance}")
        return True

    def withdraw(self):
        # 练习
        pass

In [4]:
accounts = {}
# 以类作为模板，结合具体数据，创建对象
# 创建对象，参数传给__init__
account = Account("张三", accounts)
# 调用方法时，Python自动把account传给self
account.deposit(100)
accounts['001'].get_balance()

账户创建成功: 张三 (001)
存款成功！余额: 100.0


100.0


### 2.3 关键概念

#### 2.3.1 字段（Field）
字段是类中存储数据的变量。

```python
class Account:
    
    # 类字段（所有账户共享）
    bank_name = "中国银行"  # 所有账户都属于同一银行
    
    def __init__(self, name, accounts, initial_deposit=0.0, timestamp=None):

        # 实例字段：每个对象独立
        self.account_num = f"{len(accounts) + 1:03d}"  # 账户号
        self.name = name                               # 户主姓名
        self.__balance = 0.0                           # 私有字段（余额）
        ...
```

类字段 vs 实例字段：
- 类字段：所有对象共享，通过`类名.字段名`访问
- 实例字段：每个对象独立，通过`对象.字段名`访问，一般在`__init__`方法中定义

##### `__init__`方法

这是**初始化方法**，在创建对象时自动调用。什么代码该放在`__init__`里？
- 字段的初始化
- 传入参数的合理性检查
- 初始状态设置，比如设置默认值、状态标志等
- 必要的初始计算，比如根据传入参数计算初始状态

```python
# 创建对象，参数传给__init__, 并初始化字段
# 注意：无需手动调用__init__，Python会自动调用，且无需传递self参数
account = Account("张三", accounts)
```

##### `self`的含义

`self`表示**对象自身**。我们可以把每个对象想象成一个“数据盒子”，里面存放了与这个对象相关的各类数据，比如各种实例字段和方法。那么`self`就像一个指向这个盒子的指针，我们可以通过`self`来访问和操作这个盒子中的数据。事实上，`self`只是一个约定俗成的名称，也可以用其他名称代替（如`this`），但建议使用`self`。

In [6]:
class AccountDemo:
    def __init__(this, account_number):
        this.account_number = account_number

acct = AccountDemo("123456")
print(acct)
print(acct.account_number)

123456


#### 2.3.2 方法（Method）

方法是类中定义的函数，用于操作类或对象内部的数据。与普通函数相比，方法的定义有以下区别：
+ 定义在类的代码块中，与类相关联。
+ 形参列表中首个位置一般为一个固有的参数，比如`self`，用于引用调用该方法的对象自身。

方法按照可以操作的数据类型，一般分为三种：
+ 实例方法（Instance Method）：用于操作实例级别的数据；第一个参数必须是`self`，表示对象自身。比如`__init__`方法。
+ 类方法（Class Method）：用于操作类级别的数据，通过`@classmethod`装饰器定义；第一个参数必须是`cls`，表示类本身。
+ 静态方法（Static Method）：用于完成一些与类相关的、但是无需访问类或对象内部数据的功能，通过`@staticmethod`装饰器定义；没有第一个参数。

类`Account`的`deposit`方法是一个实例方法，用于操作对象级别的数据（即账户余额）。
```python
class Account:
    ...
    def deposit(self, amount):
        ...
        self.__balance += amount  # 操作对象的字段
|
account.deposit(100)
# 调用方法时，Python自动把account传给self
# 等价于：Account.deposit(account, 100)
```


In [7]:
Account.deposit(account, 100)

存款成功！余额: 200.0


True



#### 2.3.3 **私有字段和方法**
以`__`开头的字段/方法是私有的，外部不能直接访问。



In [ ]:
# print(account.__balance)  # 错误！不能访问私有字段
# print(account.name)
# account.__record_transaction("存款", 100)  # 错误！不能调用私有方法

张三



### 2.4 总结

1. **类是一种自定义数据类型**，它定义了数据的结构（字段）和操作（方法）
2. **对象是类的实例**，是具体的数据实体
3. **`self`** 表示对象自身，是方法的第一个参数
4. **`__init__`** 是初始化方法，用于初始化对象
5. **字段**存储数据，**方法**操作数据
6. **封装**私有属性保护数据完整性

面向过程VS面向对象对比：

| 方面 | 面向过程 | 面向对象 |
|------|---------|---------|
| **与现实对应** | 不直观 | 直接映射现实实体 |
| **代码组织** | 数据与函数分离 | 数据与函数封装在一起 |
| **代码使用** | 函数可能被误用 | 类提供了清晰的接口 |
| **数据保护** | 数据可被任意修改 | 通过私有属性保护 |


### 练习建议

1. 完成`Account`类的`withdraw`方法，添加取款功能
2. 为`Account`类添加`transfer`方法，支持账户间转账


## 第三部分：面向对象编程：进阶

我们将`Account`类的相关代码组织到自定义的模块`bank_oop.py`中，并加入一些新功能。下面演示了使用效果。

In [9]:
from bank_oop import Account

In [10]:
Account.empty_accounts()
accounts = {}
# 1. 创建账户对象
account1 = Account("张三", 1000.0)
account2 = Account("李四", 500.0)
# 2. 存款操作
print("\n--- 存款操作 ---")
account1.deposit(500.50)  # 成功
account1.deposit(-1)       # 失败：金额必须大于0
# 3. 转账操作
print("\n--- 转账操作 ---")
account1.transfer(account2, 300.0)
# 4. 显示交易记录
print("\n--- 显示交易记录 ---")
account1.display_transactions()
account2.display_transactions()

所有账户对象已清空
账户创建成功: 张三 (001)
账户创建成功: 李四 (002)

--- 存款操作 ---
存款成功！余额: 1500.5
错误：存款金额必须大于0

--- 转账操作 ---
转账成功！张三 (001) 转账 300.0 到 李四 (002)

--- 显示交易记录 ---
张三 (001) 的交易记录:
2026-01-08 00:35:06 开户 1000.0 余额: 1000.0 
2026-01-08 00:35:06 存款 500.5 余额: 1500.5 
2026-01-08 00:35:06 转账 -300.0 余额: 1200.5 target_account: 002
李四 (002) 的交易记录:
2026-01-08 00:35:06 开户 500.0 余额: 500.0 
2026-01-08 00:35:06 转账 300.0 余额: 800.0 source_account: 001


### 3.1 类属性与__new__方法

通过定义`Account`的私有类属性`__accounts`并自定义`__new__`方法，实现自动分配账号并维护银行账户列表的功能。

#### 3.1.1 类属性的定义与作用

```python
class Account:
    ...
    __accounts = {}        # 所有账户都存储在这里
```

- `__accounts`：私有类属性，用于存储所有创建的账户对象，键为账户号，值为账户对象。

#### 3.1.2 账户号的自动分配

```python
def __new__(cls, *args, **kwargs):
    # 1. 调用父类的 __new__ 方法创建对象
    instance = super().__new__(cls)
    
    # 2. 自定义操作：生成并设置账户号
    account_num = f"{len(cls.__accounts) + 1:03d}"  # 格式：001, 002, ...
    instance.account_num = account_num

    # 3. 自定义操作：将新创建的实例添加到私有类字段 __accounts 中
    cls.__accounts[account_num] = instance 

    # 4. 返回创建的对象
    return instance
```

账户号的自动分配通过`__new__`方法实现，这既是一个类方法也是一个特殊方法，负责创建并返回一个新的实例。

- `__new__`方法负责将存储对象数据的”数据盒子“创建出来
- `__init__`方法负责初始化这个”数据盒子“，设置对象的初始状态


实现机制：
- 使用类属性`__accounts`的长度来确定下一个账户号
- 通过格式化字符串`f"{len(cls.__accounts) + 1:03d}"`生成3位数字的账户号（如001, 002）
- 将生成的账户号赋值给新创建的实例
- 将新实例添加到`__accounts`类属性中进行统一管理

深入理解`__new__`和`__init__`的关系

In [11]:
Account('张三', 1000.0)
Account.__init__(Account.__new__(Account), '张三', 1000.0)
# 1. 调用 Account.__new__(Account) 创建一个新的 Account 实例，但尚未初始化
# 2. 立即把刚得到的“裸”实例传给 Account.__init__
# 3. Account.__init__ 对这个实例进行初始化，设置姓名为 '张三'，额外数据为空字典 {}
# 4. 最终效果等价于 Account('张三', 1000.0)，但绕过了正常的构造流程，直接手动拆分成 __new__ 和 __init__ 两步
print(Account.get_accounts())

账户创建成功: 张三 (003)
账户创建成功: 张三 (004)
{'001': Account(张三, 001, 现金余额：1200.5), '002': Account(李四, 002, 现金余额：800.0), '003': Account(张三, 003, 现金余额：1000.0), '004': Account(张三, 004, 现金余额：1000.0)}


通过定义`__repr__`方法，我们可以自定义对象的字符串表示，方便调试和显示。
```python
    def __repr__(self):
        """
        自定义 __repr__ 方法
        返回对象的字符串表示，用于调试和显示
        """
        return f"Account({self.name}, {self.account_num}, 现金余额：{self.__balance})"
```

In [12]:
account1

Account(张三, 001, 现金余额：1200.5)

通过定义类方法`get_accounts`和`empty_accounts`，可以方便地获取当前所有账户和清空所有账户。

In [13]:
print(Account.get_accounts())
Account.empty_accounts()
print(Account.get_accounts())

{'001': Account(张三, 001, 现金余额：1200.5), '002': Account(李四, 002, 现金余额：800.0), '003': Account(张三, 003, 现金余额：1000.0), '004': Account(张三, 004, 现金余额：1000.0)}
所有账户对象已清空
{}


### 3.2 继承：理财金账户

通过继承，我们可以定义一个新账户类型，即理财金账户`WealthManagementAccount`。它继承自`Account`，并添加了下述额外功能。
- 支持理财产品的申购、赎回
- 支持理财产品净值的查询和更新
- 支持额外展示银行账户内与理财金账户相关的信息，如当前持仓、净值等

In [14]:
from bank_oop import WealthManagementAccount

# 创建一个理财金账户
print("=== 创建理财金账户 ===")
wealth_acc = WealthManagementAccount("李四", 10000.0)
print(wealth_acc)

# 购买理财产品
print("=== 购买理财产品 ===")
wealth_acc.buy_fund(5000.0)
print(wealth_acc)

# 更新理财产品净值
print("=== 更新理财产品净值 ===")
wealth_acc.update_fund_nav(1.05)
print(wealth_acc)

# 赎回部分理财产品
print("=== 赎回部分理财产品 ===")
wealth_acc.redeem_fund(2000.0)
print(wealth_acc)

# 查看交易记录
print("=== 查看交易记录 ===")
wealth_acc.display_transactions()

# 存款
print("=== 存款 ===")
wealth_acc.deposit(3000.0)
print(wealth_acc)

# 全部赎回理财产品
print("=== 全部赎回理财产品 ===")
wealth_acc.redeem_fund(wealth_acc.get_fund_shares())
print(wealth_acc)

# 再次查看交易记录
print("=== 再次查看交易记录 ===")
wealth_acc.display_transactions()

=== 创建理财金账户 ===
账户创建成功: 李四 (001)
WealthManagementAccount(李四, 001, 现金余额: 10000.00, 理财份额: 0.0000, 理财净值: 1.0000, 总价值: 10000.00)
=== 购买理财产品 ===
购买成功！花费 5000.0，获得 5000.0000 份额，当前净值: 1.0
WealthManagementAccount(李四, 001, 现金余额: 5000.00, 理财份额: 5000.0000, 理财净值: 1.0000, 总价值: 10000.00)
=== 更新理财产品净值 ===
理财产品净值已更新：1.0 → 1.05
WealthManagementAccount(李四, 001, 现金余额: 5000.00, 理财份额: 5000.0000, 理财净值: 1.0500, 总价值: 10250.00)
=== 赎回部分理财产品 ===
赎回成功！赎回 2000.0000 份额，获得 2100.00，当前净值: 1.05
WealthManagementAccount(李四, 001, 现金余额: 7100.00, 理财份额: 3000.0000, 理财净值: 1.0500, 总价值: 10250.00)
=== 查看交易记录 ===
李四 (001) 的交易记录:
2026-01-08 00:35:06 开户 10000.0 余额: 10000.0 
2026-01-08 00:35:06 购买理财 5000.0 余额: 5000.0 fund_shares: 5000.0 fund_nav: 1.0 total_value: 10000.0 shares: 5000.0
2026-01-08 00:35:06 净值更新 0 余额: 5000.0 fund_shares: 5000.0 fund_nav: 1.05 total_value: 10250.0 old_nav: 1.0 new_nav: 1.05
2026-01-08 00:35:06 赎回理财 2100.0 余额: 7100.0 fund_shares: 3000.0 fund_nav: 1.05 total_value: 10250.0 shares: 2000.0
=== 存款 ===
存款成功！

In [15]:
from pprint import pprint
print("=== 查看所有账户 ===")
pprint(Account.get_accounts())

=== 查看所有账户 ===
{'001': WealthManagementAccount(李四, 001, 现金余额: 13250.00, 理财份额: 0.0000, 理财净值: 1.0500, 总价值: 13250.00)}
